# vepyr — Build Ensembl VEP Cache

Download an Ensembl VEP offline cache and convert it to optimized Parquet files and fjall KV stores.

In [1]:
import logging

logging.basicConfig(level=logging.INFO)

In [2]:
import os

os.environ["RUST_LOG"] = "info"

In [3]:
from vepyr import build_cache

In [ ]:
!pip install ipywidgets jupyter tqdm

In [4]:
# Configure
RELEASE = 115
CACHE_DIR = "/tmp/vepyr_cache"
SPECIES = "homo_sapiens"
ASSEMBLY = "GRCh38"
CACHE_TYPE = "vep"  # "vep", "merged", or "refseq"


# Set to an existing unpacked VEP cache directory to skip download, e.g.:
# LOCAL_CACHE = "/path/to/homo_sapiens/115_GRCh38"
LOCAL_CACHE = None

In [6]:
results = build_cache(
    release=RELEASE,
    cache_dir=CACHE_DIR,
    species=SPECIES,
    assembly=ASSEMBLY,
    cache_type=CACHE_TYPE,
    local_cache=LOCAL_CACHE,
    partitions=8,
)

INFO:datafusion_bio_function_vep.cache_builder:variation: parquet exists, building fjall from existing parquet files
INFO:datafusion_bio_function_vep.cache_builder:variation: rebuilding fjall from 26 parquet files
INFO:fjall.db:Creating database at /tmp/vepyr_cache/parquet/115_GRCh38_vep/variation.fjall
INFO:lsm_tree.tree.ingest:Finished ingestion writer
INFO:lsm_tree.tree.ingest:Finished ingestion writer
INFO:datafusion_bio_function_vep.cache_builder:variation: registered 438 non-canonical contigs for fjall rebuild
INFO:datafusion_bio_function_vep.cache_builder:zstd dict trained: 9164 samples, dict 114688 bytes
INFO:lsm_tree.tree.ingest:Finished ingestion writer
INFO:datafusion_bio_function_vep.cache_builder:variation.fjall: chr1 ingested in 107.9s (1/26 files, 83219731 positions)
INFO:lsm_tree.tree.ingest:Finished ingestion writer
INFO:datafusion_bio_function_vep.cache_builder:variation.fjall: chr2 ingested in 113.7s (2/26 files, 171528489 positions)
INFO:lsm_tree.tree.ingest:Finishe

In [ ]:
import os

for path, rows in results:
    size_mb = os.path.getsize(path) / (1024 * 1024)
    name = os.path.basename(path)
    print(f"{name:45s} {rows:>12,} rows  {size_mb:8.1f} MB")

In [ ]:
import pyarrow.parquet as pq

# Inspect one of the generated files
variation_file = [p for p, _ in results if "variation" in p][0]
table = pq.read_table(variation_file)
print(f"Schema: {table.schema}")
print(f"Rows: {table.num_rows:,}")
table.to_pandas().head()